# Lab 8: Azure Monitor & Application Insights for RAG

**Duration:** 30 minutes  
**Environment:** Azure ML Studio Notebook  

---

## Learning Objectives

- Enable Application Insights for the RAG pipeline
- Log custom metrics: latency, token usage, search quality
- Create alerts for anomalous behavior
- Build a cost tracking dashboard

## Prerequisites

Labs 1-3 completed

---

© 2026 Great Learning. All rights reserved.

## Step 1: Load Configuration

In [ ]:
import os, json, urllib.request, ssl, subprocess

RESOURCE_GROUP = 'rg-promptflow-rag-lab'

# Auto-detect OpenAI resource that has BOTH gpt-4o and text-embedding-3-small deployments
# This handles the case where Lab 1 was run multiple times creating multiple OpenAI resources
print('Detecting Azure resources...')
r = subprocess.run(['az', 'cognitiveservices', 'account', 'list', '-g', RESOURCE_GROUP,
    '--query', "[?kind=='OpenAI'].name", '-o', 'tsv'], capture_output=True, text=True)
openai_candidates = [n.strip() for n in r.stdout.strip().split('\n') if n.strip()]

OPENAI_NAME = ''
for candidate in openai_candidates:
    # Check if this resource has both required deployments
    r = subprocess.run(['az', 'cognitiveservices', 'account', 'deployment', 'list',
        '-n', candidate, '-g', RESOURCE_GROUP, '--query', '[].name', '-o', 'tsv'],
        capture_output=True, text=True)
    deployments = r.stdout.strip().split('\n')
    if 'gpt-4o' in deployments and 'text-embedding-3-small' in deployments:
        OPENAI_NAME = candidate
        print(f'  Found OpenAI with deployments: {candidate}')
        break

if not OPENAI_NAME:
    print('ERROR: No OpenAI resource found with both gpt-4o and text-embedding-3-small deployments.')
    print(f'  Candidates checked: {openai_candidates}')
    print('  Fix: Go back to Lab 1 and run Steps 4 + 5 to create deployments.')
    raise SystemExit(1)

# Auto-detect Search service
r = subprocess.run(['az', 'search', 'service', 'list', '-g', RESOURCE_GROUP,
    '--query', '[].name', '-o', 'tsv'], capture_output=True, text=True)
search_candidates = [n.strip() for n in r.stdout.strip().split('\n') if n.strip()]
SEARCH_NAME = search_candidates[0] if search_candidates else ''

# Auto-detect Storage account
r = subprocess.run(['az', 'storage', 'account', 'list', '-g', RESOURCE_GROUP,
    '--query', '[0].name', '-o', 'tsv'], capture_output=True, text=True)
STORAGE_NAME = r.stdout.strip()

if not SEARCH_NAME:
    print('ERROR: No Search service found. Run Lab 1 first.')
    raise SystemExit(1)

# Get endpoints and keys
r = subprocess.run(['az', 'cognitiveservices', 'account', 'show', '-n', OPENAI_NAME, '-g', RESOURCE_GROUP,
    '--query', 'properties.endpoint', '-o', 'tsv'], capture_output=True, text=True)
OPENAI_ENDPOINT = r.stdout.strip()
if not OPENAI_ENDPOINT.endswith('/'): OPENAI_ENDPOINT += '/'

r = subprocess.run(['az', 'cognitiveservices', 'account', 'keys', 'list', '-n', OPENAI_NAME, '-g', RESOURCE_GROUP,
    '--query', 'key1', '-o', 'tsv'], capture_output=True, text=True)
OPENAI_KEY = r.stdout.strip()

SEARCH_ENDPOINT = f'https://{SEARCH_NAME}.search.windows.net'
r = subprocess.run(['az', 'search', 'admin-key', 'show', '--service-name', SEARCH_NAME, '-g', RESOURCE_GROUP,
    '--query', 'primaryKey', '-o', 'tsv'], capture_output=True, text=True)
SEARCH_KEY = r.stdout.strip()

ctx = ssl.create_default_context()

print(f'\n✅ Resources detected:')
print(f'  OpenAI:  {OPENAI_NAME}')
print(f'  Search:  {SEARCH_NAME}')
print(f'  Storage: {STORAGE_NAME}')
print(f'  Endpoint: {OPENAI_ENDPOINT}')

## Azure CLI Authentication

Azure ML compute instances have a **managed identity** — this logs in instantly with no browser needed.

> If managed identity fails (permissions not assigned), the cell falls back to device code login.

In [ ]:
import subprocess, json as _j

# Check if already logged in
r = subprocess.run(['az', 'account', 'show'], capture_output=True, text=True)
if r.returncode == 0:
    acct = _j.loads(r.stdout)
    print(f'✅ Already logged in: {acct.get("user", {}).get("name", "unknown")}')
    print(f'   Subscription: {acct.get("name", "unknown")}')
else:
    # Try managed identity first (instant on Azure ML compute)
    r2 = subprocess.run(['az', 'login', '--identity'], capture_output=True, text=True)
    if r2.returncode == 0:
        acct = _j.loads(r2.stdout)[0]
        print(f'✅ Logged in via managed identity')
        print(f'   Subscription: {acct.get("name", "unknown")}')
    else:
        print('Managed identity not available. Using device code login...')
        !az login --use-device-code

## Step 2: Create Application Insights

In [ ]:
!az monitor app-insights component create \
    --app {APPINSIGHTS_NAME} \
    --location {LOCATION} \
    --resource-group {RESOURCE_GROUP} \
    --kind web \
    --application-type web \
    --output table

# Get instrumentation key
result = subprocess.run(['az','monitor','app-insights','component','show',
    '--app',APPINSIGHTS_NAME,'--resource-group',RESOURCE_GROUP,
    '--query','instrumentationKey','-o','tsv'], capture_output=True, text=True)
APPINSIGHTS_KEY = result.stdout.strip()
print(f'\n✅ Application Insights created.')
print(f'   Key: {APPINSIGHTS_KEY[:15]}...')

## Step 3: RAG Pipeline with Telemetry

Run 8 banking queries and capture latency, token usage, and cost per query.

In [ ]:
class RAGTelemetry:
    def __init__(self):
        self.events = []
    def log(self, name, props, metrics):
        self.events.append({'name':name,'ts':time.time(),'props':props,'metrics':metrics})

telemetry = RAGTelemetry()

def call_openai(messages, max_tokens=150, temperature=0.1):
    url = f"{OPENAI_ENDPOINT}openai/deployments/gpt-4o/chat/completions?api-version=2024-06-01"
    data = json.dumps({'messages':messages,'max_tokens':max_tokens,'temperature':temperature}).encode()
    req = urllib.request.Request(url, data=data, headers={'Content-Type':'application/json','api-key':OPENAI_KEY})
    start = time.time()
    with urllib.request.urlopen(req, context=ctx) as resp:
        result = json.loads(resp.read())
    return result['choices'][0]['message']['content'], result.get('usage',{}), time.time()-start

def get_embedding(text):
    url = f"{OPENAI_ENDPOINT}openai/deployments/text-embedding-3-small/embeddings?api-version=2024-06-01"
    data = json.dumps({'input':text}).encode()
    req = urllib.request.Request(url, data=data, headers={'Content-Type':'application/json','api-key':OPENAI_KEY})
    start = time.time()
    with urllib.request.urlopen(req, context=ctx) as resp:
        result = json.loads(resp.read())
    return result['data'][0]['embedding'], time.time()-start

def hybrid_search(query, top_k=3):
    vector, embed_lat = get_embedding(query)
    url = f"{SEARCH_ENDPOINT}/indexes/banking-policies/docs/search?api-version=2024-07-01"
    body = {'search':query,'vectorQueries':[{'vector':vector,'k':top_k,'fields':'content_vector','kind':'vector'}],
            'select':'title,content,source_file','top':top_k}
    data = json.dumps(body).encode()
    req = urllib.request.Request(url, data=data, method='POST',
        headers={'Content-Type':'application/json','api-key':SEARCH_KEY})
    start = time.time()
    with urllib.request.urlopen(req, context=ctx) as resp:
        results = json.loads(resp.read()).get('value', [])
    return results, embed_lat, time.time()-start

def rag_with_telemetry(question, rid):
    t0 = time.time()
    results, embed_lat, search_lat = hybrid_search(question)
    context = '\n\n'.join([f'[{r["source_file"]}] {r["content"]}' for r in results])
    telemetry.log('Search', {'rid':rid,'q':question}, {'embed_ms':round(embed_lat*1000,1),'search_ms':round(search_lat*1000,1)})
    answer, usage, gen_lat = call_openai([{'role':'system','content':'Banking assistant. Answer from context. Cite sources.'},
        {'role':'user','content':f'Context:\n{context}\n\nQuestion: {question}'}])
    total = time.time()-t0
    pt, ct, tt = usage.get('prompt_tokens',0), usage.get('completion_tokens',0), usage.get('total_tokens',0)
    cost = (pt*0.005 + ct*0.015)/1000
    telemetry.log('RAGComplete', {'rid':rid,'q':question},
        {'total_ms':round(total*1000,1),'gen_ms':round(gen_lat*1000,1),'tokens':tt,'cost_usd':round(cost,6)})
    return answer, total, tt, cost

print('✅ Telemetry-enabled RAG pipeline ready.')

## Step 4: Run Test Queries with Telemetry

In [ ]:
test_queries = [
    'What is the wire transfer limit for personal accounts?',
    'How do I dispute a credit card transaction?',
    'What savings account has the highest APY?',
    'What should I do if I suspect fraud?',
    'What are the personal loan interest rates?',
    'Is dual authorization required for business transfers?',
    'What is the early withdrawal penalty for a 12-month CD?',
    'How long does a fraud investigation take?',
]

print('=' * 70)
print('  RAG PIPELINE TELEMETRY')
print('=' * 70)

total_cost = 0
total_tokens_all = 0
latencies = []

for i, q in enumerate(test_queries, 1):
    answer, latency, tokens, cost = rag_with_telemetry(q, f'req-{i:03d}')
    total_cost += cost
    total_tokens_all += tokens
    latencies.append(latency)
    print(f'  [{i}] {q[:50]}...')
    print(f'      Latency: {latency*1000:.0f}ms | Tokens: {tokens} | Cost: ${cost:.5f}')

## Step 5: Observability Dashboard

In [ ]:
print('=' * 70)
print('  📊 OBSERVABILITY DASHBOARD')
print('=' * 70)

print(f'\n  📈 Request Metrics:')
print(f'     Total requests:     {len(test_queries)}')
print(f'     Avg latency:        {sum(latencies)/len(latencies)*1000:.0f}ms')
print(f'     P50 latency:        {sorted(latencies)[len(latencies)//2]*1000:.0f}ms')
print(f'     P95 latency:        {sorted(latencies)[int(len(latencies)*0.95)]*1000:.0f}ms')
print(f'     Max latency:        {max(latencies)*1000:.0f}ms')

print(f'\n  💰 Cost Metrics:')
print(f'     Total tokens:       {total_tokens_all}')
print(f'     Total cost:         ${total_cost:.5f}')
print(f'     Avg cost/query:     ${total_cost/len(test_queries):.5f}')
print(f'     Est. monthly (10K): ${total_cost/len(test_queries)*10000:.2f}')

print(f'\n  🚨 Anomaly Detection:')
avg_lat = sum(latencies)/len(latencies)
anomalies = [(i,q,l) for i,(q,l) in enumerate(zip(test_queries,latencies),1) if l > avg_lat*2]
if anomalies:
    for i,q,l in anomalies:
        print(f'     ⚠️  Slow query [{i}]: {l*1000:.0f}ms (>{avg_lat*2*1000:.0f}ms threshold)')
else:
    print(f'     ✅ No anomalies (threshold: {avg_lat*2*1000:.0f}ms)')

print(f'\n  📋 Sample KQL Queries for Azure Monitor:')
print(f'     customMetrics | where name == "total_latency_ms" | summarize avg(value) by bin(timestamp, 1h)')
print(f'     customMetrics | where name == "total_tokens" | summarize sum(value) by bin(timestamp, 1h)')
print(f'     customEvents | where todouble(customMeasurements.total_latency_ms) > 3000')

## Step 6: Search Service Metrics

In [ ]:
url = f"{SEARCH_ENDPOINT}/indexes/banking-policies/stats?api-version=2024-07-01"
req = urllib.request.Request(url, headers={'api-key':SEARCH_KEY})
with urllib.request.urlopen(req, context=ctx) as resp:
    stats = json.loads(resp.read())

print('📊 Search Index Statistics:')
print(f'   Documents indexed: {stats.get("documentCount", "N/A")}')
print(f'   Storage used:      {stats.get("storageSize", 0) / 1024:.1f} KB')
print(f'   Vector index size: {stats.get("vectorIndexSize", 0) / 1024:.1f} KB')

## ✅ Lab 8 Complete

**What you accomplished:**
- Created Application Insights for telemetry
- Logged custom metrics: latency, tokens, cost per query
- Built observability dashboard with P50/P95 latencies
- Implemented anomaly detection for slow queries
- Generated KQL query examples for Azure Monitor

**Next:** Open `promptflow_lab9_keyvault_security.ipynb`


> 🧹 **Cleanup:** Run the cleanup cell in **Lab 9** when all labs are complete to delete all resources.
---

© 2026 Great Learning. All rights reserved.